# HDB Data Understanding
Import the core table `data/hdb.csv`, check data quality, standardise variable formats and generate descriptive statistics.
MRT distances and inflation adjustments are not yet available.

## Variable definitions and units
- **`year`**: registration year, for example `2017`.
- **`month`**: registration month number from `1` to `12`. The original `2017-01` period becomes `year = 2017` and `month = 1` in the cleaned table.
- **`storey_range`**: an **ordered categorical (ordinal) variable** identifying a floor band. `01 TO 03` means floor 1, 2 or 3; `10 TO 12` means floor 10, 11 or 12. The exact floor is not recorded. `storey_lower` and `storey_upper` are the interval endpoints. The bands are ordered from low to high in Python; they do not provide exact floor measurements.
- **`resale_price`**: total resale price of the flat in **Singapore dollars (SGD)**. For example, `232000` means **S$232,000**. Prices are nominal and have not been adjusted for inflation.

The input table retains the source period for tracing records. The cleaned table uses `year` and numeric `month`. Display labels show units while CSV price values stay numeric.
Source: [HDB resale transaction dataset](https://data.gov.sg/datasets/d_8b84c4ee58e3cfc0ece0d773c8ca6abc/view).

## 1. Import data
Read block numbers as text to preserve letters and leading zeros.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 25)
pd.set_option("display.float_format", "{:,.2f}".format)

# Locate the dataset when starting from the notebook or a parent project folder.
candidates = [Path.cwd(), Path.cwd() / "HDB_Project_Starter",
              Path.cwd() / "HDB_Project_Starter" / "HDB_Project_Starter"]
PROJECT_DIR = next((p for p in candidates if (p / "data/hdb.csv").is_file()), None)
if PROJECT_DIR is None:
    raise FileNotFoundError("Please run this notebook from the project folder containing data/hdb.csv.")

DATA_DIR = PROJECT_DIR / "data"
df = pd.read_csv(DATA_DIR / "hdb.csv", dtype={"block": "string"})
print(f"Rows: {df.shape[0]:,} | Columns: {df.shape[1]}")
display(df.head().rename(columns={"month": "source_month (YYYY-MM)", "resale_price": "resale_price (SGD)"}))
df.info()

Rows: 240,919 | Columns: 21


,source_row,source_month (YYYY-MM),town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,remaining_lease,resale_price (SGD),address_key,transaction_year,transaction_month_number,remaining_lease_months,remaining_lease_years,storey_lower,storey_upper,log_nominal_resale_price,same_public_fields_flag
0,2,2017-01,ANG MO KIO,2 ROOM,406,ANG MO KIO AVE 10,10 TO 12,44.00,Improved,1979,61 years 04 months,"232,000.00",406 ANG MO KIO AVE 10,2017,1,736,61.33,10,12,12.35,False
1,3,2017-01,ANG MO KIO,3 ROOM,108,ANG MO KIO AVE 4,01 TO 03,67.00,New Generation,1978,60 years 07 months,"250,000.00",108 ANG MO KIO AVE 4,2017,1,727,60.58,1,3,12.43,False
2,4,2017-01,ANG MO KIO,3 ROOM,602,ANG MO KIO AVE 5,01 TO 03,67.00,New Generation,1980,62 years 05 months,"262,000.00",602 ANG MO KIO AVE 5,2017,1,749,62.42,1,3,12.48,False
3,5,2017-01,ANG MO KIO,3 ROOM,465,ANG MO KIO AVE 10,04 TO 06,68.00,New Generation,1980,62 years 01 month,"265,000.00",465 ANG MO KIO AVE 10,2017,1,745,62.08,4,6,12.49,False
4,6,2017-01,ANG MO KIO,3 ROOM,601,ANG MO KIO AVE 5,01 TO 03,67.00,New Generation,1980,62 years 05 months,"265,000.00",601 ANG MO KIO AVE 5,2017,1,749,62.42,1,3,12.49,False


<class 'pandas.DataFrame'>
RangeIndex: 240919 entries, 0 to 240918
Data columns (total 21 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   source_row                240919 non-null  int64  
 1   month                     240919 non-null  str    
 2   town                      240919 non-null  str    
 3   flat_type                 240919 non-null  str    
 4   block                     240919 non-null  string 
 5   street_name               240919 non-null  str    
 6   storey_range              240919 non-null  str    
 7   floor_area_sqm            240919 non-null  float64
 8   flat_model                240919 non-null  str    
 9   lease_commence_date       240919 non-null  int64  
 10  remaining_lease           240919 non-null  str    
 11  resale_price              240919 non-null  float64
 12  address_key               240919 non-null  str    
 13  transaction_year          240919 non-null  int64  
 14 

## 2. Check data quality
Check missing values and records with identical public fields. Retain these records because unit numbers and unique transaction IDs are unavailable.

In [2]:
# Check both null values and whitespace-only strings.
check_df = df.replace(r"^\s*$", pd.NA, regex=True)
missing = pd.DataFrame({
    "Missing count": check_df.isna().sum(),
    "Missing (%)": check_df.isna().mean() * 100
})
display(missing)

,Missing count,Missing (%)
source_row,0,0.00
month,0,0.00
town,0,0.00
flat_type,0,0.00
block,0,0.00
street_name,0,0.00
storey_range,0,0.00
floor_area_sqm,0,0.00
flat_model,0,0.00
lease_commence_date,0,0.00


In [3]:
RAW_FIELDS = [
    "month", "town", "flat_type", "block", "street_name", "storey_range",
    "floor_area_sqm", "flat_model", "lease_commence_date", "remaining_lease", "resale_price"
]

# Use original public fields: source_row is different for every record.
same_fields = df.duplicated(subset=RAW_FIELDS, keep=False)
print(f"Rows with identical public fields: {same_fields.sum():,}")
print(f"Rows that would be removed by deduplication: {df.duplicated(subset=RAW_FIELDS).sum():,}")
display(df.loc[same_fields, ["source_row"] + RAW_FIELDS].head())

Rows with identical public fields: 635
Rows that would be removed by deduplication: 318


,source_row,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,remaining_lease,resale_price
224,226,2017-01,BUKIT MERAH,4 ROOM,106,HENDERSON CRES,07 TO 09,81.00,Improved,1975,57 years,"470,000.00"
243,245,2017-01,BUKIT MERAH,4 ROOM,106,HENDERSON CRES,07 TO 09,81.00,Improved,1975,57 years,"470,000.00"
304,306,2017-01,CENTRAL AREA,3 ROOM,271,QUEEN ST,16 TO 18,68.00,Improved,1979,61 years 02 months,"470,000.00"
305,307,2017-01,CENTRAL AREA,3 ROOM,271,QUEEN ST,16 TO 18,68.00,Improved,1979,61 years 02 months,"470,000.00"
505,507,2017-01,JURONG EAST,4 ROOM,265,TOH GUAN RD,04 TO 06,101.00,Model A,1998,80 years 09 months,"470,000.00"


## 3. Clean data
Trim text, collapse repeated spaces, standardise numeric types, and check dates, leases, storeys and log prices.
Retain all records without automatic outlier removal or missing-value imputation. Invalid formats or values require review.
The cleaned table uses separate integer `year` and `month` columns. Redundant date columns are removed. Storey ranges are stored as an ordered categorical variable in Python.

In [4]:
df_clean = df.copy()

text_cols = ["month", "town", "flat_type", "block", "street_name",
             "storey_range", "flat_model", "remaining_lease", "address_key"]
for col in text_cols:
    df_clean[col] = (df_clean[col].astype("string").str.strip()
                     .str.replace(r"\s+", " ", regex=True).replace("", pd.NA))

integer_cols = ["source_row", "lease_commence_date", "transaction_year",
                "transaction_month_number", "remaining_lease_months", "storey_lower", "storey_upper"]
float_cols = ["floor_area_sqm", "resale_price", "remaining_lease_years", "log_nominal_resale_price"]
for col in integer_cols:
    df_clean[col] = pd.to_numeric(df_clean[col], errors="raise").astype("Int64")
for col in float_cols:
    df_clean[col] = pd.to_numeric(df_clean[col], errors="raise").astype("float64")

# Missing or invalid values require review rather than automatic imputation or deletion.
assert not df_clean.isna().any().any(), "Missing values need review."
assert np.isfinite(df_clean[integer_cols + float_cols].to_numpy(dtype=float)).all(), "Non-finite values need review."
assert df_clean["source_row"].is_unique, "Source row numbers must be unique."
assert df_clean["resale_price"].gt(0).all(), "Check nonpositive resale prices."
assert df_clean["floor_area_sqm"].gt(0).all(), "Check nonpositive floor areas."

transaction_month = pd.to_datetime(df_clean["month"], format="%Y-%m", errors="raise")
assert df_clean["transaction_year"].eq(transaction_month.dt.year).all()
assert df_clean["transaction_month_number"].eq(transaction_month.dt.month).all()
assert df_clean["lease_commence_date"].le(df_clean["transaction_year"]).all()

lease = df_clean["remaining_lease"].str.extract(r"^(\d+) years?(?: (\d+) months?)?$")
assert lease[0].notna().all(), "Check remaining lease format."
lease_years = pd.to_numeric(lease[0], errors="raise")
lease_months = pd.to_numeric(lease[1].fillna("0"), errors="raise")
assert lease_months.between(0, 11).all()
assert df_clean["remaining_lease_months"].eq(lease_years * 12 + lease_months).all()
assert df_clean["remaining_lease_months"].gt(0).all()
assert np.allclose(df_clean["remaining_lease_years"], df_clean["remaining_lease_months"] / 12)

storey = df_clean["storey_range"].str.extract(r"^(\d+) TO (\d+)$")
assert storey.notna().all().all(), "Check storey range format."
assert df_clean["storey_lower"].eq(pd.to_numeric(storey[0])).all()
assert df_clean["storey_upper"].eq(pd.to_numeric(storey[1])).all()
assert df_clean["storey_lower"].le(df_clean["storey_upper"]).all()
assert np.allclose(df_clean["log_nominal_resale_price"], np.log(df_clean["resale_price"]))
assert df_clean["address_key"].eq(df_clean["block"].str.upper() + " " + df_clean["street_name"].str.upper()).all()

# Recalculate the identical-public-fields flag and retain all records.
df_clean["same_public_fields_flag"] = df_clean.duplicated(subset=RAW_FIELDS, keep=False)
assert len(df_clean) == len(df)
# Replace the combined source period and redundant aliases with year and month.
df_clean["year"] = transaction_month.dt.year.astype("Int64")
df_clean["month"] = transaction_month.dt.month.astype("Int64")
df_clean = df_clean.drop(columns=["transaction_year", "transaction_month_number"])
first_cols = ["source_row", "year", "month"]
df_clean = df_clean[first_cols + [c for c in df_clean.columns if c not in first_cols]]

# Floor bands have a natural order, but the exact floor is unavailable.
storey_levels = (df_clean[["storey_range", "storey_lower"]].drop_duplicates()
                 .sort_values("storey_lower")["storey_range"].tolist())
df_clean["storey_range"] = pd.Categorical(df_clean["storey_range"],
                                         categories=storey_levels, ordered=True)
print("Data checks passed. All records retained.")

Data checks passed. All records retained.


In [5]:
cleaning_summary = pd.DataFrame({
    "Before": [len(df), df.shape[1], int(check_df.isna().sum().sum())],
    "After": [len(df_clean), df_clean.shape[1], int(df_clean.isna().sum().sum())]
}, index=["Rows", "Columns", "Missing cells"])
display(cleaning_summary)

,Before,After
Rows,240919,240919
Columns,21,20
Missing cells,0,0


## 4. Describe data
Summarise numeric variables using count, mean, standard deviation, minimum, quartiles and maximum.
Prices are in SGD, floor area is in square metres and remaining lease is in years. Row numbers are excluded from the analysis variables.

In [6]:
numeric_cols = ["resale_price", "floor_area_sqm", "remaining_lease_years",
                "lease_commence_date", "log_nominal_resale_price"]
display(df_clean[numeric_cols].describe().T.rename(index={
    "resale_price": "resale_price (SGD)",
    "floor_area_sqm": "floor_area (sqm)",
    "remaining_lease_years": "remaining_lease (years)"
}))

,count,mean,std,min,25%,50%,75%,max
resale_price (SGD),"240,919.00","535,254.50","192,568.62","140,000.00","390,000.00","505,000.00","642,000.00","1,728,000.00"
floor_area (sqm),"240,919.00",96.69,24.02,31.00,81.00,93.00,112.00,366.70
remaining_lease (years),"240,919.00",74.07,14.33,39.33,62.17,73.75,88.50,97.75
lease_commence_date,"240,919.00","1,996.62",14.39,"1,966.00","1,985.00","1,997.00","2,012.00","2,023.00"
log_nominal_resale_price,"240,919.00",13.13,0.35,11.85,12.87,13.13,13.37,14.36


In [7]:
# Categorical variables: number of categories and most frequent category.
category_cols = ["town", "flat_type", "flat_model", "storey_range"]
display(df_clean[category_cols].describe().T)

# Flat-type distribution.
flat_type_counts = df_clean["flat_type"].value_counts()
flat_type_summary = pd.DataFrame({
    "Count": flat_type_counts,
    "Percent (%)": flat_type_counts / len(df_clean) * 100
})
display(flat_type_summary.round(2))

# Ordered floor-band distribution.
storey_counts = df_clean["storey_range"].value_counts(sort=False)
storey_summary = pd.DataFrame({"Count": storey_counts,
                               "Percent (%)": storey_counts / len(df_clean) * 100})
display(storey_summary.round(2))

,count,unique,top,freq
town,240919,26,SENGKANG,19489
flat_type,240919,7,4 ROOM,102385
flat_model,240919,21,Model A,86901
storey_range,240919,17,04 TO 06,55239


,Count,Percent (%)
flat_type,,
4 ROOM,102385,42.50
5 ROOM,58940,24.46
3 ROOM,57237,23.76
EXECUTIVE,17120,7.11
2 ROOM,5060,2.10
MULTI-GENERATION,89,0.04
1 ROOM,88,0.04


,Count,Percent (%)
storey_range,,
01 TO 03,42271,17.55
04 TO 06,55239,22.93
07 TO 09,50488,20.96
10 TO 12,44983,18.67
13 TO 15,23272,9.66
16 TO 18,10918,4.53
19 TO 21,4724,1.96
22 TO 24,3290,1.37
25 TO 27,2058,0.85


In [8]:
# Annual record counts and months covered; 2026 is an incomplete year.
year_summary = df_clean.groupby("year").agg(
    Transactions=("source_row", "size"),
    Months_covered=("month", "nunique")
)
display(year_summary)
registration_period = pd.to_datetime(df_clean[["year", "month"]].assign(day=1))
print(f"Period: {registration_period.min():%Y-%m} to {registration_period.max():%Y-%m}")
print(f"Unique addresses: {df_clean['address_key'].nunique():,}")

,Transactions,Months_covered
year,,
2017,20509,12
2018,21561,12
2019,22186,12
2020,23333,12
2021,29087,12
2022,26720,12
2023,25754,12
2024,27832,12
2025,25085,12


Period: 2017-01 to 2026-09
Unique addresses: 9,749


## 5. Save cleaned data
Save as `data/hdb_clean.csv` and retain the input table `data/hdb.csv`.
Use `hdb_clean.csv` for subsequent basic analysis. Add MRT distances before analyses involving MRT accessibility.
The saved table has 20 columns, including separate `year` and `month`. Prices remain numeric in SGD. CSV stores floor-band labels; their ordering is reapplied when the file is read.

In [9]:
OUTPUT_PATH = DATA_DIR / "hdb_clean.csv"
df_clean.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")

# Read the saved file to check row counts and block identifiers.
saved = pd.read_csv(OUTPUT_PATH, dtype={"block": "string"})
assert len(saved) == len(df_clean)
assert saved.columns.tolist() == df_clean.columns.tolist()
assert saved["block"].tolist() == df_clean["block"].tolist()
print(f"Saved: data/{OUTPUT_PATH.name} | Rows: {len(saved):,}")

Saved: data/hdb_clean.csv | Rows: 240,919


## 6. View cleaned data
Read `data/hdb_clean.csv` to inspect the saved analysis table. This section can also be run independently after restarting the kernel.
MRT distances and inflation adjustments have not yet been added.

In [10]:
from pathlib import Path
import pandas as pd

# Find the saved dataset from the notebook or workspace folder.
folders = [Path.cwd(), Path.cwd() / "HDB_Project_Starter",
           Path.cwd() / "HDB_Project_Starter" / "HDB_Project_Starter"]
clean_path = next((folder / "data/hdb_clean.csv" for folder in folders
                   if (folder / "data/hdb_clean.csv").is_file()), None)
if clean_path is None:
    raise FileNotFoundError("Cannot find data/hdb_clean.csv. Check the notebook working directory.")

clean_data = pd.read_csv(clean_path, dtype={"block": "string"})
# Restore the ordered category after reading the CSV.
storey_levels = sorted(clean_data["storey_range"].unique(),
                       key=lambda label: int(label.split(" TO ")[0]))
clean_data["storey_range"] = pd.Categorical(clean_data["storey_range"],
                                          categories=storey_levels, ordered=True)
preview_cols = ["year", "month", "town", "flat_type", "resale_price", "storey_range",
                "floor_area_sqm", "remaining_lease_years", "block", "street_name"]
display_labels = {"resale_price": "resale_price (SGD)",
                  "storey_range": "storey_range (floors)",
                  "floor_area_sqm": "floor_area (sqm)",
                  "remaining_lease_years": "remaining_lease (years)"}
print(f"Cleaned dataset: {clean_data.shape[0]:,} rows, {clean_data.shape[1]} columns")
display(clean_data[preview_cols].head(10).rename(columns=display_labels))

Cleaned dataset: 240,919 rows, 20 columns


,year,month,town,flat_type,resale_price (SGD),storey_range (floors),floor_area (sqm),remaining_lease (years),block,street_name
0,2017,1,ANG MO KIO,2 ROOM,"232,000.00",10 TO 12,44.00,61.33,406,ANG MO KIO AVE 10
1,2017,1,ANG MO KIO,3 ROOM,"250,000.00",01 TO 03,67.00,60.58,108,ANG MO KIO AVE 4
2,2017,1,ANG MO KIO,3 ROOM,"262,000.00",01 TO 03,67.00,62.42,602,ANG MO KIO AVE 5
3,2017,1,ANG MO KIO,3 ROOM,"265,000.00",04 TO 06,68.00,62.08,465,ANG MO KIO AVE 10
4,2017,1,ANG MO KIO,3 ROOM,"265,000.00",01 TO 03,67.00,62.42,601,ANG MO KIO AVE 5
5,2017,1,ANG MO KIO,3 ROOM,"275,000.00",01 TO 03,68.00,63.00,150,ANG MO KIO AVE 5
6,2017,1,ANG MO KIO,3 ROOM,"280,000.00",04 TO 06,68.00,61.50,447,ANG MO KIO AVE 10
7,2017,1,ANG MO KIO,3 ROOM,"285,000.00",04 TO 06,67.00,58.33,218,ANG MO KIO AVE 1
8,2017,1,ANG MO KIO,3 ROOM,"285,000.00",04 TO 06,68.00,61.50,447,ANG MO KIO AVE 10
9,2017,1,ANG MO KIO,3 ROOM,"285,000.00",01 TO 03,67.00,61.33,571,ANG MO KIO AVE 3


In [11]:
# Column names, data types, missing values and measurement units.
units = {"year": "Calendar year", "month": "Month number (1-12)",
         "storey_range": "Ordered floor band; exact floor unavailable",
         "storey_lower": "Floor number", "storey_upper": "Floor number",
         "resale_price": "SGD; total price per flat",
         "floor_area_sqm": "Square metres", "remaining_lease_years": "Years",
         "remaining_lease_months": "Months", "lease_commence_date": "Calendar year",
         "log_nominal_resale_price": "Natural log of nominal price in SGD"}
column_summary = pd.DataFrame({"Data type": clean_data.dtypes.astype(str),
                               "Missing values": clean_data.isna().sum()})
column_summary["Unit / meaning"] = [units.get(col, "") for col in column_summary.index]
display(column_summary)

# Summary statistics with units shown in the row labels.
display(clean_data[["resale_price", "floor_area_sqm", "remaining_lease_years"]]
        .describe().T.rename(index=display_labels).round(2))

,Data type,Missing values,Unit / meaning
source_row,int64,0,
year,int64,0,Calendar year
month,int64,0,Month number (1-12)
town,str,0,
flat_type,str,0,
block,string,0,
street_name,str,0,
storey_range,category,0,Ordered floor band; exact floor unavailable
floor_area_sqm,float64,0,Square metres
flat_model,str,0,


,count,mean,std,min,25%,50%,75%,max
resale_price (SGD),"240,919.00","535,254.50","192,568.62","140,000.00","390,000.00","505,000.00","642,000.00","1,728,000.00"
floor_area (sqm),"240,919.00",96.69,24.02,31.00,81.00,93.00,112.00,366.70
remaining_lease (years),"240,919.00",74.07,14.33,39.33,62.17,73.75,88.50,97.75


### Browse a group
Change the town and flat type below to inspect a different group. The filtered view does not change the saved dataset.

In [12]:
selected_town = "ANG MO KIO"
selected_flat_type = "4 ROOM"

clean_view = clean_data.loc[
    clean_data["town"].eq(selected_town)
    & clean_data["flat_type"].eq(selected_flat_type),
    preview_cols
]
print(f"{selected_town} | {selected_flat_type}: {len(clean_view):,} records")
display(clean_view.head(10).rename(columns=display_labels))

ANG MO KIO | 4 ROOM: 2,815 records


,year,month,town,flat_type,resale_price (SGD),storey_range (floors),floor_area (sqm),remaining_lease (years),block,street_name
33,2017,1,ANG MO KIO,4 ROOM,"400,000.00",10 TO 12,92.00,61.50,472,ANG MO KIO AVE 10
34,2017,1,ANG MO KIO,4 ROOM,"400,000.00",07 TO 09,91.00,61.50,475,ANG MO KIO AVE 10
35,2017,1,ANG MO KIO,4 ROOM,"403,000.00",01 TO 03,94.00,63.75,629,ANG MO KIO AVE 4
36,2017,1,ANG MO KIO,4 ROOM,"410,000.00",01 TO 03,92.00,63.00,546,ANG MO KIO AVE 10
37,2017,1,ANG MO KIO,4 ROOM,"425,888.00",01 TO 03,98.00,61.08,131,ANG MO KIO AVE 3
38,2017,1,ANG MO KIO,4 ROOM,"430,000.00",04 TO 06,97.00,59.50,254,ANG MO KIO AVE 4
39,2017,1,ANG MO KIO,4 ROOM,"435,000.00",04 TO 06,92.00,61.83,470,ANG MO KIO AVE 10
40,2017,1,ANG MO KIO,4 ROOM,"435,000.00",04 TO 06,91.00,62.42,601,ANG MO KIO AVE 5
41,2017,1,ANG MO KIO,4 ROOM,"440,000.00",04 TO 06,92.00,62.08,463,ANG MO KIO AVE 10
42,2017,1,ANG MO KIO,4 ROOM,"460,000.00",04 TO 06,97.00,58.50,207,ANG MO KIO AVE 1
